# **Phase 3 - Big data context** (DuckDB vs Spark / Hadoop)

In [1]:
import duckdb, sqlite3, time, os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import csv

**3A - Engine Setup (SQLite Pre-load)**

In [2]:
CSV_PATH = "flights.csv"
DB_PATH = "flights_sqlite.db"

if not os.path.exists(DB_PATH):
    print("Loading CSV into SQLite (one-time setup)...")
    con = sqlite3.connect(DB_PATH)
    con.execute("DROP TABLE IF EXISTS flights")
    con.execute("""CREATE TABLE flights (
        YEAR INT, MONTH INT, DAY INT, DAY_OF_WEEK INT,
        AIRLINE TEXT, FLIGHT_NUMBER INT, TAIL_NUMBER TEXT,
        ORIGIN_AIRPORT TEXT, DESTINATION_AIRPORT TEXT,
        SCHEDULED_DEPARTURE INT, DEPARTURE_TIME REAL,
        DEPARTURE_DELAY REAL, TAXI_OUT REAL, WHEELS_OFF REAL,
        SCHEDULED_TIME REAL, ELAPSED_TIME REAL, AIR_TIME REAL,
        DISTANCE INT, WHEELS_ON REAL, TAXI_IN REAL,
        SCHEDULED_ARRIVAL INT, ARRIVAL_TIME REAL,
        ARRIVAL_DELAY REAL, DIVERTED INT, CANCELLED INT,
        CANCELLATION_REASON TEXT, AIR_SYSTEM_DELAY REAL,
        SECURITY_DELAY REAL, AIRLINE_DELAY REAL,
        LATE_AIRCRAFT_DELAY REAL, WEATHER_DELAY REAL)""")

    with open(CSV_PATH, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        data = []                          # renamed from 'rows' to avoid clash
        for r in reader:
            data.append((
                r['YEAR'], r['MONTH'], r['DAY'], r['DAY_OF_WEEK'],
                r['AIRLINE'], r['FLIGHT_NUMBER'], r['TAIL_NUMBER'],
                r['ORIGIN_AIRPORT'], r['DESTINATION_AIRPORT'],
                r['SCHEDULED_DEPARTURE'], r['DEPARTURE_TIME'],
                r['DEPARTURE_DELAY'], r['TAXI_OUT'], r['WHEELS_OFF'],
                r['SCHEDULED_TIME'], r['ELAPSED_TIME'], r['AIR_TIME'],
                r['DISTANCE'], r['WHEELS_ON'], r['TAXI_IN'],
                r['SCHEDULED_ARRIVAL'], r['ARRIVAL_TIME'],
                r['ARRIVAL_DELAY'], r['DIVERTED'], r['CANCELLED'],
                r['CANCELLATION_REASON'], r['AIR_SYSTEM_DELAY'],
                r['SECURITY_DELAY'], r['AIRLINE_DELAY'],
                r['LATE_AIRCRAFT_DELAY'], r['WEATHER_DELAY']
            ))

    con.executemany(
        "INSERT INTO flights VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
        data
    )
    con.commit()
    con.close()
    print(f"SQLite DB ready. {len(data)} rows loaded.")
else:
    print("SQLite DB already exists, skipping load.")

SQLite DB already exists, skipping load.


**3A (Fix) - SQLite Rebuild (Empty DB Correction)**

In [3]:
CSV_PATH = "flights.csv"
DB_PATH = "flights_sqlite.db"

# Force delete and rebuild
os.remove(DB_PATH)
print("Old DB deleted. Rebuilding...")

con = sqlite3.connect(DB_PATH)
con.execute("CREATE TABLE flights (YEAR INT, MONTH INT, DAY INT, DAY_OF_WEEK INT, AIRLINE TEXT, FLIGHT_NUMBER INT, TAIL_NUMBER TEXT, ORIGIN_AIRPORT TEXT, DESTINATION_AIRPORT TEXT, SCHEDULED_DEPARTURE INT, DEPARTURE_TIME REAL, DEPARTURE_DELAY REAL, TAXI_OUT REAL, WHEELS_OFF REAL, SCHEDULED_TIME REAL, ELAPSED_TIME REAL, AIR_TIME REAL, DISTANCE INT, WHEELS_ON REAL, TAXI_IN REAL, SCHEDULED_ARRIVAL INT, ARRIVAL_TIME REAL, ARRIVAL_DELAY REAL, DIVERTED INT, CANCELLED INT, CANCELLATION_REASON TEXT, AIR_SYSTEM_DELAY REAL, SECURITY_DELAY REAL, AIRLINE_DELAY REAL, LATE_AIRCRAFT_DELAY REAL, WEATHER_DELAY REAL)")

with open(CSV_PATH, encoding='utf-8') as f:
    reader = csv.DictReader(f)
    data = []
    for r in reader:
        data.append((
            r['YEAR'], r['MONTH'], r['DAY'], r['DAY_OF_WEEK'],
            r['AIRLINE'], r['FLIGHT_NUMBER'], r['TAIL_NUMBER'],
            r['ORIGIN_AIRPORT'], r['DESTINATION_AIRPORT'],
            r['SCHEDULED_DEPARTURE'], r['DEPARTURE_TIME'],
            r['DEPARTURE_DELAY'], r['TAXI_OUT'], r['WHEELS_OFF'],
            r['SCHEDULED_TIME'], r['ELAPSED_TIME'], r['AIR_TIME'],
            r['DISTANCE'], r['WHEELS_ON'], r['TAXI_IN'],
            r['SCHEDULED_ARRIVAL'], r['ARRIVAL_TIME'],
            r['ARRIVAL_DELAY'], r['DIVERTED'], r['CANCELLED'],
            r['CANCELLATION_REASON'], r['AIR_SYSTEM_DELAY'],
            r['SECURITY_DELAY'], r['AIRLINE_DELAY'],
            r['LATE_AIRCRAFT_DELAY'], r['WEATHER_DELAY']
        ))

con.executemany("INSERT INTO flights VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", data)
con.commit()

count = con.execute("SELECT COUNT(*) FROM flights").fetchone()[0]
con.close()
print(f"Done. {count:,} rows loaded.")

Old DB deleted. Rebuilding...
Done. 5,819,079 rows loaded.


**3B - Three-Engine Benchmark (DuckDB vs SQLite vs Pandas)**

In [4]:
CSV_PATH = "flights.csv"
DB_PATH = "flights_sqlite.db"

query_aggregation = """
    SELECT AIRLINE, count(*) as delayed_flights, avg(DEPARTURE_DELAY) as avg_delay
    FROM flights
    WHERE DEPARTURE_DELAY > 15
    GROUP BY AIRLINE
    ORDER BY delayed_flights DESC
"""

# ── 1. DuckDB ──────────────────────────────────────────────
t0 = time.time()
duckdb.sql(query_aggregation.replace("FROM flights", "FROM read_csv_auto('flights.csv')")).fetchall()
duck_time = time.time() - t0
print(f"DuckDB:  {duck_time:.3f}s")

# ── 2. SQLite — verify row count first ────────────────────
con = sqlite3.connect(DB_PATH)
row_count = con.execute("SELECT COUNT(*) FROM flights").fetchone()[0]
print(f"SQLite DB has {row_count:,} rows")  # should be ~5.8M

t0 = time.time()
con.execute("""
    SELECT AIRLINE, count(*) as delayed_flights, avg(DEPARTURE_DELAY) as avg_delay
    FROM flights
    WHERE DEPARTURE_DELAY > 15
    GROUP BY AIRLINE
    ORDER BY delayed_flights DESC
""").fetchall()
sqlite_time = time.time() - t0
con.close()
print(f"SQLite:  {sqlite_time:.3f}s")

# ── 3. Pandas (row-oriented, no vectorized SQL execution) ──
t0 = time.time()
df = pd.read_csv(CSV_PATH, low_memory=False)
result = (df[df['DEPARTURE_DELAY'] > 15]
            .groupby('AIRLINE')
            .agg(delayed_flights=('DEPARTURE_DELAY', 'count'),
                 avg_delay=('DEPARTURE_DELAY', 'mean'))
            .sort_values('delayed_flights', ascending=False))
pandas_time = time.time() - t0
print(f"Pandas:  {pandas_time:.3f}s")

# ── Summary ────────────────────────────────────────────────
print("\n── Benchmark Summary ──────────────────────────────")
print(f"{'Engine':<10} {'Time':>8}  {'vs DuckDB':>10}  {'Architecture'}")
print(f"{'─'*55}")
print(f"{'DuckDB':<10} {duck_time:>7.3f}s  {'1.0×':>10}  Columnar vectorized, in-process")
print(f"{'SQLite':<10} {sqlite_time:>7.3f}s  {sqlite_time/duck_time:>9.1f}×  Row-based, no vectorization")
print(f"{'Pandas':<10} {pandas_time:>7.3f}s  {pandas_time/duck_time:>9.1f}×  Row-oriented, interpreted Python")

DuckDB:  3.186s
SQLite DB has 5,819,079 rows
SQLite:  2.347s
Pandas:  41.935s

── Benchmark Summary ──────────────────────────────
Engine         Time   vs DuckDB  Architecture
───────────────────────────────────────────────────────
DuckDB       3.186s        1.0×  Columnar vectorized, in-process
SQLite       2.347s        0.7×  Row-based, no vectorization
Pandas      41.935s       13.2×  Row-oriented, interpreted Python


**3C - Fair Comparison (DuckDB Pre-loaded vs SQLite Pre-loaded)**

In [5]:
# Pre-load into a persistent DuckDB database file (one-time)
con = duckdb.connect("flights.duckdb")
con.execute("CREATE TABLE IF NOT EXISTS flights AS SELECT * FROM read_csv_auto('flights.csv')")
con.close()

# Now time the query on pre-loaded data — apples to apples with SQLite
con = duckdb.connect("flights.duckdb")
t0 = time.time()
con.execute("""
    SELECT AIRLINE, count(*) as delayed_flights, avg(DEPARTURE_DELAY) as avg_delay
    FROM flights
    WHERE DEPARTURE_DELAY > 15
    GROUP BY AIRLINE
    ORDER BY delayed_flights DESC
""").fetchall()
duck_preloaded = time.time() - t0
con.close()
print(f"DuckDB (pre-loaded): {duck_preloaded:.3f}s")
print(f"SQLite (pre-loaded): 2.186s")
print(f"Ratio: {2.186/duck_preloaded:.1f}× — DuckDB advantage on equal footing")

DuckDB (pre-loaded): 0.065s
SQLite (pre-loaded): 2.186s
Ratio: 33.5× — DuckDB advantage on equal footing
